# Plot figure showing published projects

NOTE!!: Does dimensions tap out at 50,000 records? It seems so

## Set up

In [ ]:
import os
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
from tableone import TableOne
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# path to datasets
base_path = os.path.join("..", "data", "physionet")

In [ ]:
patents_filename = "Dimensions-Patent-2025-11-20_16-49-09.csv"
publications_filename = "Dimensions-Publication-2025-11-20_16-48-49.csv"

## Load the data

In [ ]:
# Custom function to parse the datetime
def parse_publish_date(date_str):
    try:
        # Try the format with microseconds first
        return datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S.%f%z")
    except ValueError:
        # If that fails, try the format without microseconds
        return datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S%z")

In [ ]:
# iterate folder
for file in os.listdir(base_path):
    print(file)

### Patents

In [ ]:
patents = pd.read_csv(os.path.join(base_path, patents_filename), encoding="utf8", low_memory=False)
patents.head()

In [ ]:
patents.columns

In [ ]:
# Create a full range of years from 1999 to 2025
full_years = pd.DataFrame({'Granted year': range(1999, 2026)})

# Count journals and rename columns
granted_patents = patents['Granted year'].value_counts().reset_index()
granted_patents.columns = ['Granted year', 'counts']

# Merge full_years with granted_patents to include missing years
granted_patents = pd.merge(full_years, granted_patents, on='Granted year', how='left')

# Replace NaN values with 0 for missing years
granted_patents['counts'] = granted_patents['counts'].fillna(0)

granted_patents

### Publications

In [ ]:
publications = pd.read_csv(os.path.join(base_path, publications_filename), encoding="utf8", low_memory=False)
publications.head()

In [ ]:
publications.columns

In [ ]:
# Filter publications to between 1999 and 2026
publications = publications.loc[(publications['PubYear'] >= 1999) & (publications['PubYear'] <= 2026)]

In [ ]:
# Create a full range of years from 1999 to 2025
full_years = pd.DataFrame({'PubYear': range(1999, 2026)})

# Count journals and rename columns
final_publications = publications['PubYear'].value_counts().reset_index()
final_publications.columns = ['PubYear', 'counts']

# Merge full_years with granted_patents to include missing years
final_publications = pd.merge(full_years, final_publications, on='PubYear', how='left')

# Replace NaN values with 0 for missing years
final_publications['counts'] = final_publications['counts'].fillna(0)

# granted_patents


# # count publications and rename columns
# publications = publications['PubYear'].value_counts()
# publications = publications.to_frame().reset_index().rename(columns= {"index": 'PubYear'})
# publications.index.name = 'index'

In [ ]:
final_publications.head()

## Plot functions

In [ ]:
def plot_publications_patents_modern(
        publications,
        granted_patents,
        size_title=34,
        size_axes_labels=34,
        size_tick_labels=26,
        incomplete=True,
        bold=False):

    # Colors + fonts (same as the modern style)
    font_type = "Arial Black" if bold else "Arial"
    accent = "#4c72b0"
    accent_faded = "rgba(76,114,176,0.35)"
    text_col = "#222"
    tick_col = "#555"

    # ----------------------------------------------
    # Subplots
    # ----------------------------------------------
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            "Articles Referencing PhysioNet per Year",
            "Patents Granted per Year"
        ),
        horizontal_spacing=0.12
    )

    # ----------------------------------------------
    # Helper: add star above final bar
    # ----------------------------------------------
    def add_star(fig, years, counts, row, col, star_size=18):
        last_x = years.iloc[-1]
        last_y = counts.iloc[-1]
        fig.add_trace(
            go.Scatter(
                x=[last_x],
                y=[last_y * 1.05],
                mode="text",
                text=["★"],
                textfont=dict(size=star_size, color="#c0392b"),
                hoverinfo="skip",
                showlegend=False
            ),
            row=row, col=col
        )

    # ----------------------------------------------
    # Prepare data
    # ----------------------------------------------
    pub_years = publications["PubYear"]
    pub_counts = publications["counts"].astype(float)

    patent_years = granted_patents["Granted year"]
    patent_counts = granted_patents["counts"].astype(float)

    padding_factor = 1.15
    max_y_publications = pub_counts.max() * padding_factor
    max_y_patents = patent_counts.max() * padding_factor

    # ----------------------------------------------
    # Plot 1: Publications
    # ----------------------------------------------
    colors_pub = [
        accent_faded if (incomplete and i == len(pub_counts) - 1) else accent
        for i in range(len(pub_counts))
    ]

    fig.add_trace(
        go.Bar(
            x=pub_years,
            y=pub_counts,
            marker=dict(color=colors_pub, line=dict(color=accent, width=0.8)),
            width=0.55,
            name="Articles",
        ),
        row=1, col=1
    )

    if incomplete:
        add_star(fig, pub_years, pub_counts, row=1, col=1)

    # ----------------------------------------------
    # Plot 2: Patents
    # ----------------------------------------------
    colors_pat = [
        accent_faded if (incomplete and i == len(patent_counts) - 1) else accent
        for i in range(len(patent_counts))
    ]

    fig.add_trace(
        go.Bar(
            x=patent_years,
            y=patent_counts,
            marker=dict(color=colors_pat, line=dict(color=accent, width=0.8)),
            width=0.55,
            name="Patents",
        ),
        row=1, col=2
    )

    if incomplete:
        add_star(fig, patent_years, patent_counts, row=1, col=2)

    # ----------------------------------------------
    # Layout
    # ----------------------------------------------
    fig.update_layout(
        template="simple_white",
        font=dict(family=font_type, size=size_tick_labels, color=text_col),
        showlegend=False,
        margin=dict(l=90, r=40, t=120, b=100),
        height=700,
        width=1400
    )

    # ----------------------------------------------
    # Center and restyle subplot titles (prevent clipping)
    # ----------------------------------------------
    fig.update_layout(
        annotations=[
            dict(
                text="Articles Referencing PhysioNet per Year",
                x=0.225, y=1.10,
                xanchor="center",
                font=dict(size=size_title, family=font_type, color=text_col),
                showarrow=False
            ),
            dict(
                text="Patents Granted per Year",
                x=0.775, y=1.10,
                xanchor="center",
                font=dict(size=size_title, family=font_type, color=text_col),
                showarrow=False
            ),
        ]
    )

    # ----------------------------------------------
    # Axes styling
    # ----------------------------------------------

    # Publications
    fig.update_xaxes(
        title_text="Year",
        title_font=dict(size=size_axes_labels, family=font_type, color=text_col),
        tickfont=dict(size=size_tick_labels, color=tick_col),
        showline=True, linecolor="#333", linewidth=1, ticks="outside",
        range=[pub_years.min() - 0.5, pub_years.max() + 0.5],
        row=1, col=1
    )
    fig.update_yaxes(
        title_text="Articles",
        title_font=dict(size=size_axes_labels, family=font_type, color=text_col),
        tickfont=dict(size=size_tick_labels, color=tick_col),
        showline=True, linecolor="#333", linewidth=1, ticks="outside",
        range=[0, max_y_publications],
    
        tickmode="array",
        tickvals=[v for v in range(0, int(max_y_publications)+1, 1000)],
        ticktext=[f"{int(v/1000)}k" if v > 0 else "0"
                  for v in range(0, int(max_y_publications)+1, 1000)],
    
        row=1, col=1
    )

    # Patents
    fig.update_xaxes(
        title_text="Year",
        title_font=dict(size=size_axes_labels, family=font_type, color=text_col),
        tickfont=dict(size=size_tick_labels, color=tick_col),
        showline=True, linecolor="#333", linewidth=1, ticks="outside",
        range=[patent_years.min() - 0.5, patent_years.max() + 0.5],
        row=1, col=2
    )
    fig.update_yaxes(
        title_text="Patents",
        title_font=dict(size=size_axes_labels, family=font_type, color=text_col),
        tickfont=dict(size=size_tick_labels, color=tick_col),
        separatethousands=True,
        showline=True, linecolor="#333", linewidth=1, ticks="outside",
        range=[0, max_y_patents],
        row=1, col=2
    )

    return fig


## Plot it

In [ ]:
# Plot publications and patents
fig = plot_publications_patents_modern(final_publications,
                                 granted_patents,
                                 incomplete=True)

fig.write_image("../figures/figure_5_citations.png")
fig.write_image("../figures/figure_5_citations.svg")